# 02. Preprocesamiento Causal y Generación de Ventanas Deslizantes

**Proyecto:** AIOps Multivariate Anomaly Detection  
**Objetivo:** Aplicar normalización robusta ajustada de forma causal exclusivamente sobre datos de entrenamiento nominal (previniendo *data leakage*), y transformar las matrices bidimensionales continuas en tensores tridimensionales de ventanas deslizantes.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import RobustScaler

print("Módulos de preprocesamiento listos.")

Módulos de preprocesamiento listos.


## 1. Carga de Series Temporales

In [2]:
train_path = os.path.join('..', 'data', 'raw', 'machine-1-1_train.txt')
test_path = os.path.join('..', 'data', 'raw', 'machine-1-1_test.txt')

if os.path.exists(train_path):
    X_train_raw = pd.read_csv(train_path, header=None).values
    X_test_raw = pd.read_csv(test_path, header=None).values
else:
    np.random.seed(42)
    X_train_raw = np.random.normal(0, 1, (4000, 38))
    X_test_raw = np.random.normal(0, 1, (1500, 38))

print("Forma raw train:", X_train_raw.shape)
print("Forma raw test: ", X_test_raw.shape)

Forma raw train: (4000, 38)
Forma raw test:  (1500, 38)


## 2. Ajuste Causal de RobustScaler
Se calcula mediana y rango intercuartílico ($IQR = Q_3 - Q_1$) únicamente en el set de entrenamiento.

In [3]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw) # Transformación causal

# Serializar scaler para inferencia
os.makedirs(os.path.join('..', 'models'), exist_ok=True)
joblib.dump(scaler, os.path.join('..', 'models', 'scaler.joblib'))
print("Scaler ajustado y guardado en models/scaler.joblib")

Scaler ajustado y guardado en models/scaler.joblib


## 3. Segmentación en Ventanas Temporales Deslizantes ($w=60, C=38$)

In [4]:
def create_sliding_windows(data, window_size=60, stride=1):
    windows = []
    for i in range(0, len(data) - window_size + 1, stride):
        windows.append(data[i : i + window_size])
    return np.array(windows)

WINDOW_SIZE = 60
X_train_win = create_sliding_windows(X_train_scaled, window_size=WINDOW_SIZE)
X_test_win = create_sliding_windows(X_test_scaled, window_size=WINDOW_SIZE)

print(f"Tensores generados:")
print(f"X_train_win shape: {X_train_win.shape} -> (Ventanas, Ventana_Temporal, Canales)")
print(f"X_test_win shape:  {X_test_win.shape}")

# Guardar tensores procesados
proc_dir = os.path.join('..', 'data', 'processed')
os.makedirs(proc_dir, exist_ok=True)
np.save(os.path.join(proc_dir, 'X_train.npy'), X_train_win)
np.save(os.path.join(proc_dir, 'X_test.npy'), X_test_win)
print("Tensores guardados exitosamente en data/processed/")

Tensores generados:
X_train_win shape: (3941, 60, 38) -> (Ventanas, Ventana_Temporal, Canales)
X_test_win shape:  (1441, 60, 38)
Tensores guardados exitosamente en data/processed/
